# 04 — Accessibilité à pied (Bordeaux Métropole)

Entrées : `insee_carreaux_200m`, `gtfs_*`, `gbfs_stations` (Files/raw/tables, sinon tables Delta).
Sorties : tables `carreaux`, `offre_existante`, `hypotheses`, `meta` + export JSON.

In [ ]:
RAYON_M = 400
GARDER_GIRONDE = False
EXCLURE_SCOLAIRE = True
TYPES_GBFS = ["citiz", "velo"]


In [ ]:
%pip install -q pyproj


In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from pyproj import Transformer
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import cKDTree

SRC = Path("/lakehouse/default/Files/raw/tables")
OUT = Path("/lakehouse/default/Files/gold")
OUT.mkdir(parents=True, exist_ok=True)

COMMUNES_BM = {
    "33003", "33004", "33013", "33032", "33039", "33056", "33063", "33065",
    "33069", "33075", "33096", "33119", "33162", "33167", "33192", "33200",
    "33249", "33273", "33281", "33312", "33318", "33376", "33434", "33449",
    "33487", "33519", "33522", "33550",
}

TABLES_DELTA = Path("/lakehouse/default/Tables/dbo") if Path("/lakehouse/default/Tables/dbo").exists() else Path("/lakehouse/default/Tables")

def lire(nom, columns=None, filters=None):
    p = SRC / f"{nom}.parquet"
    if p.exists():
        return pd.read_parquet(p, columns=columns, filters=filters)
    from deltalake import DeltaTable
    return DeltaTable(str(TABLES_DELTA / nom)).to_pandas(columns=columns, filters=filters)

TO_3035 = Transformer.from_crs("EPSG:4326", "EPSG:3035", always_xy=True)

def projeter(df, lon="lon", lat="lat"):
    x, y = TO_3035.transform(df[lon].to_numpy(float), df[lat].to_numpy(float))
    return df.assign(x=x, y=y)


## Carreaux INSEE

In [ ]:
car = lire("insee_carreaux_200m", columns=[
    "idcar_200m", "lcog_geo", "i_est_200", "ind", "men", "men_coll", "men_pauv",
    "men_prop", "ind_65_79", "ind_80p", "epsg", "x_centre", "y_centre", "lon", "lat"])
car["code_commune"] = car["lcog_geo"].str[:5]
car = (car[(car["epsg"] == 3035) & car["code_commune"].isin(COMMUNES_BM)]
       .rename(columns={"x_centre": "x", "y_centre": "y"})
       .reset_index(drop=True))
print(f"{len(car):,} carreaux | {car['ind'].sum():,.0f} habitants")


## Arrêts GTFS

In [ ]:
def mode_gtfs(rt):
    rt = pd.to_numeric(rt, errors="coerce")
    return np.select(
        [rt.isin([0, 5, 12]) | rt.between(900, 999), rt.isin([1]) | rt.between(400, 499),
         rt.isin([4]) | rt.between(1000, 1299), rt.isin([2]) | rt.between(100, 199)],
        ["tram", "metro", "bateau", "train"], "bus")

stops = lire("gtfs_stops")
if not GARDER_GIRONDE:
    stops = stops[stops["jeu_titre"].str.contains("TBM", case=False, na=False)]
ids = stops["ressource_id"].unique().tolist()
filtre = [("ressource_id", "in", ids)]

routes = lire("gtfs_routes", filters=filtre)
routes["mode"] = mode_gtfs(routes["route_type"])
if EXCLURE_SCOLAIRE:
    noms = routes.reindex(columns=["route_short_name", "route_long_name", "route_desc"]).fillna("").astype(str)
    routes = routes[~noms.agg(" ".join, axis=1).str.contains("scol", case=False)]

trips = lire("gtfs_trips", columns=["ressource_id", "trip_id", "route_id"], filters=filtre)
trips = trips.merge(routes[["ressource_id", "route_id", "mode"]], on=["ressource_id", "route_id"])
st = lire("gtfs_stop_times", columns=["ressource_id", "trip_id", "stop_id"], filters=filtre)
st = st.drop_duplicates().merge(trips[["ressource_id", "trip_id", "mode"]], on=["ressource_id", "trip_id"])

PRIORITE = ["tram", "metro", "train", "bateau", "bus"]
st["mode"] = pd.Categorical(st["mode"], categories=PRIORITE, ordered=True)
mode_arret = st.groupby(["ressource_id", "stop_id"], observed=True)["mode"].min().reset_index()

lt = pd.to_numeric(stops["location_type"], errors="coerce").fillna(0) if "location_type" in stops else 0
quais = stops[lt == 0].merge(mode_arret, on=["ressource_id", "stop_id"])
quais = projeter(quais.dropna(subset=["stop_lat", "stop_lon"]), "stop_lon", "stop_lat")

quais = quais.reset_index(drop=True)
parent = quais.get("parent_station", pd.Series(index=quais.index, dtype=str)).fillna("").astype(str)
quais["cle"] = quais["ressource_id"].astype(str) + "|" + parent

sans = np.flatnonzero(parent == "")
nom = quais["stop_name"].fillna("").str.lower().str.strip().to_numpy()[sans]
paires = cKDTree(quais.loc[sans, ["x", "y"]].to_numpy()).query_pairs(150, output_type="ndarray")
paires = paires[nom[paires[:, 0]] == nom[paires[:, 1]]]
graphe = coo_matrix((np.ones(len(paires)), (paires[:, 0], paires[:, 1])), shape=(len(sans), len(sans)))
quais.loc[sans, "cle"] = "nom|" + pd.Series(connected_components(graphe, directed=False)[1]).astype(str).to_numpy()

arrets = (quais.groupby(["cle", "mode"], observed=True)
          .agg(nom=("stop_name", "first"), x=("x", "mean"), y=("y", "mean"),
               lat=("stop_lat", "mean"), lon=("stop_lon", "mean"), operateur=("jeu_titre", "first"))
          .reset_index())
arrets["type"] = arrets["mode"].astype(str)
arrets["id_station"] = "gtfs_" + pd.util.hash_pandas_object(arrets["cle"] + arrets["type"], index=False).astype(str)
print(f"{len(quais):,} quais -> {len(arrets):,} arrêts")
print(arrets["type"].value_counts().to_string())


## Stations GBFS

In [ ]:
gb = lire("gbfs_stations")
for c in ("lat", "lon"):
    gb[c] = pd.to_numeric(gb[c], errors="coerce")
gb = gb.dropna(subset=["lat", "lon"])

est_citiz = gb["est_citiz"].astype(str).str.lower().eq("true")
FREE_FLOATING = "dott|pony|lime|bird|tier|voi|cooltra|yego"
est_velo = gb["types_vehicules"].fillna("").str.contains("bicycle") & ~gb["operateur"].fillna("").str.lower().str.contains(FREE_FLOATING)
gb["type"] = np.select([est_citiz, est_velo], ["citiz", "velo"], "autre")
gb = projeter(gb[gb["type"].isin(TYPES_GBFS)])

gb["cle"] = gb["type"] + "|" + (gb["x"] // 20).astype(int).astype(str) + "|" + (gb["y"] // 20).astype(int).astype(str)
gb = gb.drop_duplicates("cle")
stations = gb.assign(id_station="gbfs_" + gb["ressource_id"].astype(str) + "_" + gb["station_id"].astype(str))[
    ["id_station", "nom", "type", "operateur", "lat", "lon", "x", "y"]]


## Offre dans la métropole

In [ ]:
offre = pd.concat([arrets[["id_station", "nom", "type", "operateur", "lat", "lon", "x", "y"]], stations],
                  ignore_index=True)
arbre_car = cKDTree(car[["x", "y"]].to_numpy())
dist, _ = arbre_car.query(offre[["x", "y"]].to_numpy(), distance_upper_bound=300)
offre = offre[np.isfinite(dist)].reset_index(drop=True)
print(offre["type"].value_counts().to_string())


## Accessibilité par carreau

In [ ]:
pts_car = car[["x", "y"]].to_numpy()
TYPES = sorted(offre["type"].unique())
for t in TYPES:
    sous = offre.loc[offre["type"] == t, ["x", "y"]].to_numpy()
    car[f"nb_{t}"] = cKDTree(sous).query_ball_point(pts_car, RAYON_M, return_length=True)
car["nb_options"] = car[[f"nb_{t}" for t in TYPES]].sum(axis=1)
car["options_par_habitant"] = np.where(car["ind"] > 0, car["nb_options"] / car["ind"], np.nan)

rang = car.loc[car["nb_options"] > 0, "options_par_habitant"].rank(pct=True).dropna()
car["niveau_acces"] = 1
car.loc[rang.index, "niveau_acces"] = (np.ceil(rang * 4) + 1).clip(2, 5).astype(int)
car["couvert_citiz"] = car.get("nb_citiz", 0) > 0
car["donnee_masquee"] = car["i_est_200"] == 1


## Population desservie par station

In [ ]:
voisins = arbre_car.query_ball_point(offre[["x", "y"]].to_numpy(), RAYON_M)
ind = car["ind"].to_numpy()
offre["pop_desservie"] = [int(round(ind[v].sum())) for v in voisins]
_, proche = arbre_car.query(offre[["x", "y"]].to_numpy())
offre["code_commune"] = car["code_commune"].to_numpy()[proche]


## Contrôles

In [ ]:
pop = car["ind"].sum()
resume = {"au moins 1 option": (car["nb_options"] > 0)}
resume.update({t: car[f"nb_{t}"] > 0 for t in TYPES})
print(pd.Series({k: round(100 * car.loc[m, "ind"].sum() / pop, 1) for k, m in resume.items()},
                name="% habitants à 5 min").to_string())
print(car.groupby("code_commune")["couvert_citiz"].mean().sort_values().head(10).round(2).to_string())


## Écriture

In [ ]:
carreaux = car.assign(
    id_carreau=car["idcar_200m"],
    population=car["ind"].round().astype(int),
)[["id_carreau", "code_commune", "lat", "lon", "x", "y", "population", "nb_options",
   *[f"nb_{t}" for t in TYPES], "options_par_habitant", "niveau_acces", "couvert_citiz", "donnee_masquee",
   "men", "men_coll", "men_pauv", "men_prop", "ind_65_79", "ind_80p"]]

offre_existante = offre[["id_station", "nom", "type", "operateur", "code_commune", "lat", "lon", "x", "y", "pop_desservie"]]

hypotheses = pd.DataFrame([
    ("zone", "Bordeaux Métropole (28 communes)", "Choix d'équipe"),
    ("distance_marche_m", str(RAYON_M), "5 min à pied, vol d'oiseau"),
    ("reseaux_gtfs", "Gironde incluse" if GARDER_GIRONDE else "TBM seulement", "transport.data.gouv.fr"),
    ("lignes_scolaires", "exclues" if EXCLURE_SCOLAIRE else "incluses", "Filtre nom de ligne"),
    ("types_gbfs", ",".join(TYPES_GBFS), "Stations fixes seulement (Dott, Pony et autres free-floating exclus)"),
    ("population", "Filosofi 2021, carreaux 200 m", "INSEE"),
    ("donnee_masquee", "i_est_200 = 1", "Secret statistique INSEE"),
], columns=["cle", "valeur", "source"])

meta = pd.DataFrame([{"date_maj": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
                      "version": "1", "rayon_m": RAYON_M}])

TABLES = {"carreaux": carreaux, "offre_existante": offre_existante, "hypotheses": hypotheses, "meta": meta}

import pyarrow as pa
from deltalake import write_deltalake
import notebookutils
ctx = notebookutils.runtime.context
DELTA = (f"abfss://{ctx['defaultLakehouseWorkspaceId']}@onelake.dfs.fabric.microsoft.com/"
         f"{ctx['defaultLakehouseId']}/Tables" + ("/dbo" if Path("/lakehouse/default/Tables/dbo").exists() else ""))
OPTIONS = {"bearer_token": notebookutils.credentials.getToken("storage"), "use_fabric_endpoint": "true"}

for nom_t, df in TABLES.items():
    df.to_parquet(OUT / f"{nom_t}.parquet", index=False)
    write_deltalake(f"{DELTA}/{nom_t}", pa.Table.from_pandas(df, preserve_index=False),
                    mode="overwrite", schema_mode="overwrite", storage_options=OPTIONS)
    print(f"{nom_t} : {len(df):,} lignes")


## Export JSON (backend)

In [ ]:
EXPORT = OUT / "export"
EXPORT.mkdir(exist_ok=True)

c = carreaux.drop(columns=["x", "y", "men", "men_coll", "men_pauv", "men_prop", "ind_65_79", "ind_80p"]).copy()
c["population"] = c["population"].astype("Int64").mask(c["donnee_masquee"])
c["options_par_habitant"] = c["options_par_habitant"].mask(c["donnee_masquee"])

exports = {"carreaux": c, "offre_existante": offre_existante.drop(columns=["x", "y"]),
           "hypotheses": hypotheses, "meta": meta}
for nom_t, df in exports.items():
    (EXPORT / f"{nom_t}.json").write_text(df.to_json(orient="records", force_ascii=False))
print(sorted(p.name for p in EXPORT.iterdir()))
